# K03_04 – Grid Search und Hyperparameter – Studentenversion

Diese Fassung ist für die **aktive Mitarbeit im Kurs** gedacht.

Dieses Notebook ist die **Vertiefung** zu Kapitel 3.  
Wir suchen systematisch gute Hyperparameter mit `GridSearchCV`.

## Lernziele
Nach diesem Notebook können Sie:
- den Begriff **Hyperparameter** erklären
- ein kleines Parametersuchgitter definieren
- `GridSearchCV` anwenden
- die Ergebnisse der Suche interpretieren

## 1. Problem und Datensatz

Wir bleiben beim Iris-Datensatz, damit der Fokus auf der Methode liegt.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


## 2. Basismodell


In [ ]:
baseline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Baseline-Accuracy: {acc_base:.3f}")


## 3. Was ist ein Hyperparameter?

Bei k-NN ist `n_neighbors` ein typischer Hyperparameter.  
Er wird **nicht aus den Daten gelernt**, sondern von uns festgelegt.


## 4. Grid Search vorbereiten


In [ ]:
pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

param_grid = {
    "kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11],
    "kneighborsclassifier__weights": ["uniform", "distance"]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)


## Mini-Übung 1
Bevor Sie die Suche starten:  
Welche Kombination vermuten Sie als sinnvoll – wenige oder viele Nachbarn? `uniform` oder `distance`?


## 5. Grid Search durchführen


In [ ]:
grid.fit(X_train, y_train)

print("Beste Parameter:")
print(grid.best_params_)

print("\nBester mittlerer CV-Score:")
print(round(grid.best_score_, 3))


## 6. Ergebnisse als Tabelle


In [ ]:
results = pd.DataFrame(grid.cv_results_)
cols = [
    "param_kneighborsclassifier__n_neighbors",
    "param_kneighborsclassifier__weights",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]
results[cols].sort_values("rank_test_score").head(10)


## 7. Bestes Modell auf den Testdaten bewerten


In [ ]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"Test-Accuracy des besten Modells: {acc_best:.3f}")
print(f"Baseline-Accuracy: {acc_base:.3f}")


## Mini-Übung 2
1. Ist das beste Modell auf den Testdaten besser als die Baseline?
2. Warum ist es wichtig, die Hyperparametersuche **nicht** direkt auf den Testdaten zu machen?


## 8. Zusammenfassung

- Hyperparameter werden von uns vorgegeben
- `GridSearchCV` testet mehrere Kombinationen systematisch
- die Bewertung erfolgt über Kreuzvalidierung
- erst am Ende prüfen wir das beste Modell auf separaten Testdaten
